# 09_chi_square_quick_check

Notebook này **chỉ phụ trách phần kiểm định Chi-square** trong nhóm `Utilities & Evaluation` của dự án.  
Phạm vi của file 09:

- đọc `orders_base_final` từ notebook **02**
- tạo `review_label`
- chạy `scipy.stats.chi2_contingency` cho các biến phân loại chính
- bổ sung `Cramér's V`, kiểm tra giả định expected counts, và `standardized residuals`
- lưu kết quả bảng / hình / summary để dùng cho báo cáo và phụ lục

**Phạm vi không làm**
- không train lại mô hình
- không chỉnh sửa dữ liệu upstream
- không ghi đè artifact model/rule của notebook 03–08


## Cell 1 — Import thư viện cần thiết

**Mục tiêu**
- Import đúng các thư viện tối thiểu cần dùng cho notebook 09.
- Giữ notebook gọn, không thêm framework thừa.
- Chuẩn bị đủ công cụ để:
  - đọc dữ liệu processed,
  - chạy `chi2_contingency`,
  - tính `Cramér's V`,
  - vẽ biểu đồ hỗ trợ báo cáo,
  - lưu artifact kết quả.


In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from scipy.stats import chi2_contingency


## Cell 2 — Xác định thư mục project và khai báo đường dẫn file

**Mục tiêu**
- Tìm đúng root project theo cấu trúc chung của bộ notebook.
- Giữ logic tương thích với notebook 02 và 08:
  - đọc `data/processed/orders_base_final.parquet|csv`
  - lưu artifact vào `artifacts/metrics` và `artifacts/plots`
- Không đổi tên file output lõi đã dùng ổn định trong dự án:
  - `chi_square_results.csv`
  - `chi_square_summary.json`

**Đầu ra mong đợi**
- Có đủ path cho:
  - input processed table
  - output summary CSV/JSON
  - bảng chi tiết contingency/expected/residuals
  - plots phục vụ báo cáo


In [2]:
def locate_project_base():
    candidates = [Path("."), Path(".."), Path("../..")]
    for base in candidates:
        if (base / "data" / "processed").exists():
            return base.resolve()
    return Path(".").resolve()

BASE_DIR = locate_project_base()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
ARTIFACT_DIR = BASE_DIR / "artifacts"
METRIC_DIR = ARTIFACT_DIR / "metrics"
PLOT_DIR = ARTIFACT_DIR / "plots" / "chi_square"
TABLE_DIR = METRIC_DIR / "chi_square_tables"

METRIC_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

orders_base_parquet = PROCESSED_DIR / "orders_base_final.parquet"
orders_base_csv = PROCESSED_DIR / "orders_base_final.csv"

results_path = METRIC_DIR / "chi_square_results.csv"
summary_path = METRIC_DIR / "chi_square_summary.json"

display(Markdown("### Paths overview"))
display(pd.DataFrame({
    "artifact_name": [
        "orders_base_parquet",
        "orders_base_csv",
        "results_path",
        "summary_path",
        "table_dir",
        "plot_dir",
    ],
    "path": [
        str(orders_base_parquet),
        str(orders_base_csv),
        str(results_path),
        str(summary_path),
        str(TABLE_DIR),
        str(PLOT_DIR),
    ]
}))


### Paths overview

,artifact_name,path
0,orders_base_parquet,C:\Users\trant\OneDrive\Tài liệu\Documents\U...
1,orders_base_csv,C:\Users\trant\OneDrive\Tài liệu\Documents\U...
2,results_path,C:\Users\trant\OneDrive\Tài liệu\Documents\U...
3,summary_path,C:\Users\trant\OneDrive\Tài liệu\Documents\U...
4,table_dir,C:\Users\trant\OneDrive\Tài liệu\Documents\U...
5,plot_dir,C:\Users\trant\OneDrive\Tài liệu\Documents\U...


## Cell 3 — Đọc `orders_base_final` và kiểm tra cột bắt buộc

**Mục tiêu**
- Chỉ đọc đúng artifact processed do notebook 02 sinh ra.
- Kiểm tra sớm các cột bắt buộc để notebook fail rõ nếu upstream schema thay đổi.
- Tạo snapshot nhanh về shape và một vài cột quan trọng để xác nhận dữ liệu vào đúng.

**Các cột bắt buộc cho notebook 09**
- `review_score`
- `payment_type_mode`
- `customer_state`
- `main_category`

**Tiêu chí pass**
- đọc được `orders_base_final`
- không thiếu cột bắt buộc
- dữ liệu có số dòng > 0


In [3]:
if orders_base_parquet.exists():
    df = pd.read_parquet(orders_base_parquet)
    source_path = orders_base_parquet
elif orders_base_csv.exists():
    df = pd.read_csv(orders_base_csv)
    source_path = orders_base_csv
else:
    raise FileNotFoundError(
        "Không tìm thấy orders_base_final.parquet/csv trong data/processed. "
        "Cần chạy notebook 02 trước."
    )

required_cols = ["review_score", "payment_type_mode", "customer_state", "main_category"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise KeyError(f"Thiếu cột bắt buộc cho notebook 09: {missing_cols}")

display(Markdown("### Input dataset summary"))
print("Loaded from:", source_path)
print("Shape:", df.shape)

display(pd.DataFrame({
    "required_column": required_cols,
    "exists_in_orders_base_final": [c in df.columns for c in required_cols]
}))

display(Markdown("### orders_base_final sample"))
display(df[required_cols].head(10))


### Input dataset summary

Loaded from: C:\Users\trant\OneDrive\Tài liệu\Documents\UTE\BIGDATA\olist_ml_project\data\processed\orders_base_final.parquet
Shape: (99441, 52)


,required_column,exists_in_orders_base_final
0,review_score,True
1,payment_type_mode,True
2,customer_state,True
3,main_category,True


### orders_base_final sample

,review_score,payment_type_mode,customer_state,main_category
0,4.0,voucher,SP,housewares
1,4.0,boleto,BA,perfumery
2,5.0,credit_card,GO,auto
3,5.0,credit_card,RN,pet_shop
4,5.0,credit_card,SP,stationery
5,4.0,credit_card,PR,auto
6,2.0,credit_card,RS,unknown
7,5.0,credit_card,RJ,auto
8,1.0,boleto,RS,furniture_decor
9,5.0,credit_card,SP,office_furniture


## Cell 4 — Làm sạch `review_score` và tạo biến mục tiêu `review_label`

**Mục tiêu**
- Chuẩn hóa `review_score` để mọi kiểm định dùng cùng một target phân loại.
- Gộp nhãn theo logic đơn giản, nhất quán với phần còn lại của dự án:
  - `review_score >= 4` → `positive`
  - `review_score <= 3` → `negative_or_neutral`

**Lý do chọn cách gộp**
- Phù hợp hướng diễn giải business: đánh giá tốt vs không tốt
- Tránh tạo bảng chéo quá thưa khi dùng 5 mức sao nguyên bản
- Dễ kết nối với narrative classification/report

**Tiêu chí pass**
- chỉ giữ score hợp lệ từ 1 đến 5
- tạo được `review_label`
- hiển thị phân phối nhãn để kiểm tra imbalance cơ bản


In [4]:
df = df.copy()

df["review_score"] = pd.to_numeric(df["review_score"], errors="coerce")
df = df[df["review_score"].isin([1, 2, 3, 4, 5])].copy()
df["review_label"] = np.where(df["review_score"] >= 4, "positive", "negative_or_neutral")

label_counts = df["review_label"].value_counts(dropna=False)
label_ratio = (label_counts / label_counts.sum()).round(6)

display(Markdown("### Review label distribution"))
display(pd.DataFrame({
    "count": label_counts,
    "ratio": label_ratio,
    "ratio_pct": (label_ratio * 100).round(4)
}).reset_index().rename(columns={"index": "review_label"}))


### Review label distribution

,review_label,count,ratio,ratio_pct
0,positive,76045,0.770677,77.0677
1,negative_or_neutral,22628,0.229323,22.9323


## Cell 5 — Định nghĩa helper cho Chi-square, `Cramér's V`, expected-count diagnostics và residuals

**Mục tiêu**
- Gói toàn bộ logic kiểm định vào một hàm duy nhất để tái sử dụng và dễ kiểm tra.
- Không chỉ dừng ở `p_value`, mà còn thêm:
  - `Cramér's V` để đo **mức độ mạnh/yếu** của liên hệ
  - expected count diagnostics để kiểm tra giả định của Chi-square
  - `standardized residuals` để biết ô nào đóng góp mạnh vào khác biệt

**Lưu ý phương pháp**
- `p_value` chỉ cho biết **có ý nghĩa thống kê hay không**
- `Cramér's V` mới giúp nói về **mức độ mạnh/yếu**
- Nếu nhiều ô có expected count quá thấp, diễn giải phải thận trọng hơn


In [5]:
def categorize_cramers_v(value: float) -> str:
    if pd.isna(value):
        return "unknown"
    if value < 0.10:
        return "negligible"
    if value < 0.30:
        return "small"
    if value < 0.50:
        return "medium"
    return "large"


def build_interpretation_note(row: pd.Series) -> str:
    notes = []
    notes.append("Có ý nghĩa thống kê" if row["significant_at_0_05"] else "Không có ý nghĩa thống kê")
    notes.append(f"Effect size: {row['cramers_v_strength']}")
    if row["assumption_expected_lt_5_ok"] and row["assumption_expected_lt_1_ok"]:
        notes.append("Giả định expected count đạt")
    else:
        notes.append("Diễn giải thận trọng do expected count thấp")
    return " | ".join(notes)


def run_chi_square(df_input, feature_col, target_col="review_label", top_n=None):
    work = df_input[[feature_col, target_col]].copy()
    work[feature_col] = work[feature_col].fillna("unknown").astype(str).str.strip()
    work[target_col] = work[target_col].fillna("unknown").astype(str).str.strip()

    n_rows_before_top_n = len(work)
    rows_removed_by_top_n = 0

    if top_n is not None:
        top_values = work[feature_col].value_counts().head(top_n).index.tolist()
        work = work[work[feature_col].isin(top_values)].copy()
        rows_removed_by_top_n = int(n_rows_before_top_n - len(work))

    contingency = pd.crosstab(work[feature_col], work[target_col])
    chi2, p_value, dof, expected = chi2_contingency(contingency)

    expected_df = pd.DataFrame(expected, index=contingency.index, columns=contingency.columns)
    standardized_residuals = (contingency - expected_df) / np.sqrt(expected_df)

    n = contingency.to_numpy().sum()
    r, c = contingency.shape
    min_dim = min(r - 1, c - 1)
    cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 and n > 0 else np.nan

    expected_lt_5 = expected_df < 5
    expected_lt_1 = expected_df < 1

    result = {
        "feature": feature_col,
        "target": target_col,
        "n_rows": int(len(work)),
        "n_rows_before_top_n": int(n_rows_before_top_n),
        "rows_removed_by_top_n": rows_removed_by_top_n,
        "top_n_applied": top_n if top_n is not None else "all",
        "n_categories_tested": int(contingency.shape[0]),
        "degrees_of_freedom": int(dof),
        "chi2_statistic": float(chi2),
        "p_value": float(p_value),
        "significant_at_0_05": bool(p_value < 0.05),
        "cramers_v": float(cramers_v) if pd.notna(cramers_v) else np.nan,
        "cramers_v_strength": categorize_cramers_v(cramers_v),
        "expected_lt_5_count": int(expected_lt_5.sum().sum()),
        "expected_lt_5_ratio": float(expected_lt_5.mean().mean()),
        "expected_lt_1_count": int(expected_lt_1.sum().sum()),
        "expected_lt_1_ratio": float(expected_lt_1.mean().mean()),
        "assumption_expected_lt_5_ok": bool(expected_lt_5.mean().mean() <= 0.20),
        "assumption_expected_lt_1_ok": bool(expected_lt_1.sum().sum() == 0),
    }

    return result, contingency, expected_df, standardized_residuals


def plot_stacked_share(contingency_df, title, save_path):
    share_df = contingency_df.div(contingency_df.sum(axis=1), axis=0)
    ax = share_df.plot(kind="bar", stacked=True, figsize=(10, 5))
    ax.set_title(title)
    ax.set_xlabel(contingency_df.index.name or "feature_value")
    ax.set_ylabel("Within-category share")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()


def plot_residual_heatmap(residual_df, title, save_path):
    fig, ax = plt.subplots(figsize=(8, max(4, 0.4 * len(residual_df))))
    im = ax.imshow(residual_df.values, aspect="auto")
    ax.set_title(title)
    ax.set_xticks(range(len(residual_df.columns)))
    ax.set_xticklabels(residual_df.columns)
    ax.set_yticks(range(len(residual_df.index)))
    ax.set_yticklabels(residual_df.index)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

    for i in range(residual_df.shape[0]):
        for j in range(residual_df.shape[1]):
            ax.text(j, i, f"{residual_df.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Standardized residual")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()


## Cell 6 — Chạy kiểm định cho các biến phân loại chính

**Mục tiêu**
- Kiểm tra xem các biến phân loại quan trọng có liên hệ thống kê với `review_label` hay không.

**Biến được chọn**
- `payment_type_mode`: đại diện cho hành vi thanh toán
- `customer_state`: đại diện cho khác biệt địa lý
- `main_category`: đại diện cho nhóm sản phẩm chính

**Lưu ý phương pháp**
- `main_category` chỉ lấy **top 15** category để tránh bảng chéo quá thưa.
- Kết quả được diễn giải theo 3 lớp:
  - `p_value`: có ý nghĩa thống kê hay không
  - `Cramér's V`: liên hệ mạnh hay yếu
  - expected-count diagnostics: mức độ phù hợp của phép kiểm định


In [6]:
results = []
outputs = {}

tests = [
    ("payment_type_mode", None),
    ("customer_state", None),
    ("main_category", 15),
]

for feature_col, top_n in tests:
    if feature_col not in df.columns:
        continue

    result, contingency, expected_df, residuals_df = run_chi_square(
        df_input=df,
        feature_col=feature_col,
        target_col="review_label",
        top_n=top_n
    )
    outputs[feature_col] = {
        "contingency": contingency,
        "expected": expected_df,
        "residuals": residuals_df,
    }
    results.append(result)

results_df = pd.DataFrame(results)

if len(results_df) == 0:
    raise RuntimeError("Không chạy được phép kiểm định nào. Kiểm tra lại dữ liệu đầu vào.")

results_df["interpretation_note"] = results_df.apply(build_interpretation_note, axis=1)
results_df = results_df.sort_values(["chi2_statistic", "p_value"], ascending=[False, True]).reset_index(drop=True)

display(Markdown("### Chi-square results summary"))
display(results_df)


### Chi-square results summary

,feature,target,n_rows,n_rows_before_top_n,rows_removed_by_top_n,top_n_applied,n_categories_tested,degrees_of_freedom,chi2_statistic,p_value,significant_at_0_05,cramers_v,cramers_v_strength,expected_lt_5_count,expected_lt_5_ratio,expected_lt_1_count,expected_lt_1_ratio,assumption_expected_lt_5_ok,assumption_expected_lt_1_ok,interpretation_note
0,customer_state,review_label,98673,98673,0,all,27,26,633.439717,6.234455e-117,True,0.080122,negligible,0,0.000000,0,0.00,True,True,Có ý nghĩa thống kê | Effect size: negligible ...
1,main_category,review_label,77227,98673,21446,15,15,14,252.895163,7.237353e-46,True,0.057225,negligible,0,0.000000,0,0.00,True,True,Có ý nghĩa thống kê | Effect size: negligible ...
2,payment_type_mode,review_label,98673,98673,0,all,6,5,36.026314,9.383688e-07,True,0.019108,negligible,4,0.333333,3,0.25,False,False,Có ý nghĩa thống kê | Effect size: negligible ...


## Cell 7 — Hiển thị contingency table, expected table và standardized residuals

**Mục tiêu**
- Cung cấp bằng chứng chi tiết cho từng kiểm định.
- Giúp người đọc thấy rõ:
  - bảng tần suất quan sát (`contingency`)
  - bảng tần suất kỳ vọng (`expected`)
  - ô nào đóng góp mạnh vào khác biệt (`standardized residuals`)

**Lưu ý diễn giải**
- Residual dương lớn: ô xuất hiện nhiều hơn kỳ vọng
- Residual âm lớn về trị tuyệt đối: ô xuất hiện ít hơn kỳ vọng
- Với báo cáo chính, thường chỉ cần trích 1–2 bảng tiêu biểu; còn lại đưa vào phụ lục


In [7]:
for feature_col, bundle in outputs.items():
    display(Markdown(f"### Feature: `{feature_col}` — contingency table"))
    display(bundle["contingency"].head(20))

    display(Markdown(f"### Feature: `{feature_col}` — expected frequencies"))
    display(bundle["expected"].round(3).head(20))

    display(Markdown(f"### Feature: `{feature_col}` — standardized residuals"))
    display(bundle["residuals"].round(3).head(20))


### Feature: `payment_type_mode` — contingency table

review_label,negative_or_neutral,positive
payment_type_mode,,
boleto,4514,15122
credit_card,17263,58275
debit_card,315,1206
not_defined,3,0
unknown,1,0
voucher,532,1442


### Feature: `payment_type_mode` — expected frequencies

review_label,negative_or_neutral,positive
payment_type_mode,,
boleto,4502.989,15133.011
credit_card,17322.610,58215.390
debit_card,348.800,1172.200
not_defined,0.688,2.312
unknown,0.229,0.771
voucher,452.684,1521.316


### Feature: `payment_type_mode` — standardized residuals

...

review_label,negative_or_neutral,positive
main_category,,
auto,861,3010
baby,696,2150
bed_bath_table,2478,6802
computers_accessories,1606,5033
cool_stuff,715,2841
electronics,551,1972
furniture_decor,1582,4718
garden_tools,728,2729
health_beauty,1783,6962


### Feature: `main_category` — expected frequencies

review_label,negative_or_neutral,positive
main_category,,
auto,876.836,2994.164
baby,644.659,2201.341
bed_bath_table,2102.050,7177.950
computers_accessories,1503.827,5135.173
cool_stuff,805.484,2750.516
electronics,571.495,1951.505
furniture_decor,1427.038,4872.962
garden_tools,783.059,2673.941
health_beauty,1980.865,6764.135


### Feature: `main_category` — standardized residuals

review_label,negative_or_neutral,positive
main_category,,
auto,-0.535,0.289
baby,2.022,-1.094
bed_bath_table,8.200,-4.437
computers_accessories,2.635,-1.426
cool_stuff,-3.188,1.725
electronics,-0.857,0.464
furniture_decor,4.102,-2.220
garden_tools,-1.968,1.065
health_beauty,-4.446,2.406


## Cell 8 — Lưu kết quả, lưu bảng chi tiết, vẽ biểu đồ hỗ trợ và tạo summary cuối

**Mục tiêu**
- Lưu toàn bộ artifact để:
  - dùng lại cho báo cáo / phụ lục,
  - kiểm tra nhanh trong integration,
  - tránh phải rerun notebook chỉ để lấy bảng/hình.

**File lõi phải giữ ổn định**
- `chi_square_results.csv`
- `chi_square_summary.json`

**File phụ bổ sung**
- `contingency_<feature>.csv`
- `expected_<feature>.csv`
- `standardized_residuals_<feature>.csv`
- biểu đồ stacked share
- biểu đồ residual heatmap


In [8]:
plot_registry_rows = []

for feature_col, bundle in outputs.items():
    contingency_path = TABLE_DIR / f"contingency_{feature_col}.csv"
    expected_path = TABLE_DIR / f"expected_{feature_col}.csv"
    residuals_path = TABLE_DIR / f"standardized_residuals_{feature_col}.csv"

    bundle["contingency"].to_csv(contingency_path)
    bundle["expected"].to_csv(expected_path)
    bundle["residuals"].to_csv(residuals_path)

    stacked_share_path = PLOT_DIR / f"stacked_share_{feature_col}.png"
    residual_heatmap_path = PLOT_DIR / f"residual_heatmap_{feature_col}.png"

    plot_stacked_share(
        contingency_df=bundle["contingency"],
        title=f"Share of review_label by {feature_col}",
        save_path=stacked_share_path
    )
    plot_residual_heatmap(
        residual_df=bundle["residuals"],
        title=f"Standardized residuals: {feature_col} vs review_label",
        save_path=residual_heatmap_path
    )

    plot_registry_rows.append({
        "feature": feature_col,
        "contingency_csv": str(contingency_path),
        "expected_csv": str(expected_path),
        "standardized_residuals_csv": str(residuals_path),
        "stacked_share_plot": str(stacked_share_path),
        "residual_heatmap_plot": str(residual_heatmap_path),
    })

results_df.to_csv(results_path, index=False)

assumption_warning_features = results_df.loc[
    (~results_df["assumption_expected_lt_5_ok"]) | (~results_df["assumption_expected_lt_1_ok"]),
    "feature"
].tolist()

summary_payload = {
    "source_orders_base_file": str(source_path),
    "n_tests_requested": len(tests),
    "n_tests_ran": int(len(results_df)),
    "n_significant_tests": int(results_df["significant_at_0_05"].sum()),
    "top_feature_by_chi2": results_df.sort_values("chi2_statistic", ascending=False).iloc[0]["feature"],
    "top_feature_by_cramers_v": results_df.sort_values("cramers_v", ascending=False).iloc[0]["feature"],
    "assumption_warning_features": assumption_warning_features,
    "n_assumption_warning_tests": len(assumption_warning_features),
    "safe_for_stronger_interpretation_features": results_df.loc[
        (results_df["assumption_expected_lt_5_ok"]) & (results_df["assumption_expected_lt_1_ok"]),
        "feature"
    ].tolist(),
    "results_file": str(results_path),
    "table_dir": str(TABLE_DIR),
    "plot_dir": str(PLOT_DIR),
    "plot_registry": plot_registry_rows,
    "report_note": (
        "Kết quả chi-square chỉ nên diễn giải ở mức liên hệ thống kê. "
        "Khi viết báo cáo cần kết hợp p-value, Cramér's V và expected-count diagnostics."
    ),
}

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, ensure_ascii=False, indent=2)

display(Markdown("### Output files summary"))
display(pd.DataFrame(plot_registry_rows))

print("Đã lưu:", results_path)
print("Đã lưu:", summary_path)


### Output files summary

,feature,contingency_csv,expected_csv,standardized_residuals_csv,stacked_share_plot,residual_heatmap_plot
0,payment_type_mode,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,C:\Users\trant\OneDrive\Tài liệu\Documents\U...
1,customer_state,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,C:\Users\trant\OneDrive\Tài liệu\Documents\U...
2,main_category,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,C:\Users\trant\OneDrive\Tài liệu\Documents\U...,C:\Users\trant\OneDrive\Tài liệu\Documents\U...


Đã lưu: C:\Users\trant\OneDrive\Tài liệu\Documents\UTE\BIGDATA\olist_ml_project\artifacts\metrics\chi_square_results.csv
Đã lưu: C:\Users\trant\OneDrive\Tài liệu\Documents\UTE\BIGDATA\olist_ml_project\artifacts\metrics\chi_square_summary.json


## CHECK-IN — FILE 09: `09_chi_square_quick_check (1).ipynb`

### A. THIẾT LẬP BAN ĐẦU
- [x] Hoàn thành import thư viện cần thiết cho notebook chi-square
- [x] Hoàn thành cấu hình cảnh báo
- [x] Hoàn thành định vị thư mục project
- [x] Hoàn thành khai báo các đường dẫn:
  - `BASE_DIR`
  - `PROCESSED_DIR`
  - `ARTIFACT_DIR`
  - `METRIC_DIR`
  - `PLOT_DIR`
  - `TABLE_DIR`
- [x] Hoàn thành tạo thư mục output cho:
  - metrics
  - plots
  - chi-square tables
- [x] Hoàn thành khai báo đường dẫn file:
  - `orders_base_final.parquet`
  - `orders_base_final.csv`
  - `chi_square_results.csv`
  - `chi_square_summary.json`

---

### B. ĐỌC DỮ LIỆU ĐẦU VÀO
- [x] Hoàn thành đọc `orders_base_final.parquet` nếu tồn tại
- [x] Hoàn thành fallback đọc `orders_base_final.csv` nếu cần
- [x] Hoàn thành kiểm tra các cột bắt buộc:
  - `review_score`
  - `payment_type_mode`
  - `customer_state`
  - `main_category`
- [x] Hoàn thành hiển thị summary của input dataset
- [x] Hoàn thành hiển thị sample dữ liệu đầu vào

---

### C. LÀM SẠCH `review_score` VÀ TẠO BIẾN MỤC TIÊU
- [x] Hoàn thành ép kiểu numeric cho `review_score`
- [x] Hoàn thành giữ lại các giá trị hợp lệ từ 1 đến 5
- [x] Hoàn thành tạo `review_label`
- [x] Hoàn thành gộp nhãn:
  - `review_score >= 4` → `positive`
  - `review_score <= 3` → `negative_or_neutral`
- [x] Hoàn thành thống kê phân phối `review_label`
- [x] Hoàn thành hiển thị bảng phân phối nhãn

---

### D. HÀM HELPER CHO CHI-SQUARE
- [x] Hoàn thành định nghĩa hàm `categorize_cramers_v()`
- [x] Hoàn thành định nghĩa hàm `build_interpretation_note()`
- [x] Hoàn thành định nghĩa hàm `run_chi_square()`
- [x] Hoàn thành tính trong `run_chi_square()`:
  - `contingency table`
  - `expected frequencies`
  - `standardized residuals`
  - `chi2_statistic`
  - `p_value`
  - `degrees_of_freedom`
  - `cramers_v`
  - `expected_lt_5_count`
  - `expected_lt_5_ratio`
  - `expected_lt_1_count`
  - `expected_lt_1_ratio`
  - `assumption_expected_lt_5_ok`
  - `assumption_expected_lt_1_ok`

---

### E. KHAI BÁO CÁC BIẾN PHÂN LOẠI ĐỂ KIỂM ĐỊNH
- [x] Hoàn thành khai báo danh sách biến cần kiểm định:
  - `payment_type_mode`
  - `customer_state`
  - `main_category`
- [x] Hoàn thành áp dụng `top_n=15` cho `main_category`
- [x] Hoàn thành chạy kiểm định cho từng biến phân loại chính

---

### F. CHẠY KIỂM ĐỊNH CHI-SQUARE
- [x] Hoàn thành chạy chi-square cho `payment_type_mode`
- [x] Hoàn thành chạy chi-square cho `customer_state`
- [x] Hoàn thành chạy chi-square cho `main_category`
- [x] Hoàn thành lưu:
  - `contingency`
  - `expected`
  - `residuals`
  vào `outputs`
- [x] Hoàn thành tổng hợp `results`
- [x] Hoàn thành tạo `results_df`
- [x] Hoàn thành thêm cột `interpretation_note`
- [x] Hoàn thành sắp xếp kết quả theo:
  - `chi2_statistic`
  - `p_value`
- [x] Hoàn thành hiển thị bảng `Chi-square results summary`

---

### G. HIỂN THỊ BẢNG CHI TIẾT CHO TỪNG BIẾN
- [x] Hoàn thành hiển thị `contingency table` cho từng feature
- [x] Hoàn thành hiển thị `expected frequencies` cho từng feature
- [x] Hoàn thành hiển thị `standardized residuals` cho từng feature

---

### H. LƯU BẢNG CHI TIẾT CHO TỪNG BIẾN
- [x] Hoàn thành lưu `contingency_<feature>.csv`
- [x] Hoàn thành lưu `expected_<feature>.csv`
- [x] Hoàn thành lưu `standardized_residuals_<feature>.csv`
- [x] Hoàn thành lưu riêng bảng chi tiết cho:
  - `payment_type_mode`
  - `customer_state`
  - `main_category`

---

### I. VẼ BIỂU ĐỒ HỖ TRỢ
- [x] Hoàn thành tạo biểu đồ `stacked share` cho từng feature
- [x] Hoàn thành tạo biểu đồ `residual heatmap` cho từng feature
- [x] Hoàn thành lưu:
  - `stacked_share_<feature>.png`
  - `residual_heatmap_<feature>.png`

---

### J. LƯU KẾT QUẢ TỔNG HỢP
- [x] Hoàn thành lưu `chi_square_results.csv`
- [x] Hoàn thành tạo `summary_payload`
- [x] Hoàn thành tổng hợp trong `summary_payload`:
  - `source_orders_base_file`
  - `n_tests_requested`
  - `n_tests_ran`
  - `n_significant_tests`
  - `top_feature_by_chi2`
  - `top_feature_by_cramers_v`
  - `assumption_warning_features`
  - `n_assumption_warning_tests`
  - `safe_for_stronger_interpretation_features`
  - `results_file`
  - `table_dir`
  - `plot_dir`
  - `plot_registry`
  - `report_note`
- [x] Hoàn thành lưu `chi_square_summary.json`

---

### K. TỔNG HỢP FILE OUTPUT
- [x] Hoàn thành tạo `plot_registry_rows`
- [x] Hoàn thành tạo bảng `Output files summary`
- [x] Hoàn thành hiển thị danh sách file output đã lưu

---

### L. KẾT LUẬN NOTEBOOK
- [x] Hoàn thành phần tóm tắt phạm vi notebook 09
- [x] Hoàn thành ghi rõ file đầu vào notebook 09 sử dụng
- [x] Hoàn thành ghi rõ file đầu ra notebook 09 tạo ra
- [x] Hoàn thành ghi rõ các file bị ảnh hưởng
- [x] Hoàn thành ghi rõ các file không bị ảnh hưởng
- [x] Hoàn thành ghi chú diễn giải khi viết báo cáo

---

### M. KẾT QUẢ ĐẦU RA CHÍNH CỦA NOTEBOOK
- [x] Hoàn thành phần **đọc `orders_base_final`**
- [x] Hoàn thành phần **kiểm tra cột bắt buộc**
- [x] Hoàn thành phần **làm sạch `review_score`**
- [x] Hoàn thành phần **tạo `review_label`**
- [x] Hoàn thành phần **xây dựng helper cho chi-square**
- [x] Hoàn thành phần **chạy kiểm định cho các biến phân loại chính**
- [x] Hoàn thành phần **tạo contingency / expected / residual tables**
- [x] Hoàn thành phần **hiển thị bảng chi tiết cho từng feature**
- [x] Hoàn thành phần **lưu bảng chi tiết**
- [x] Hoàn thành phần **vẽ stacked share plots**
- [x] Hoàn thành phần **vẽ residual heatmaps**
- [x] Hoàn thành phần **lưu `chi_square_results.csv`**
- [x] Hoàn thành phần **lưu `chi_square_summary.json`**
- [x] Hoàn thành phần **tạo output files summary**
- [x] Hoàn thành phần **kết luận notebook 09**

### Kết luận cuối:
Notebook `09_chi_square_quick_check (1).ipynb` đã **hoàn thành** toàn bộ các hạng mục chính thuộc phạm vi **Chi-square Testing, Review Label Preparation, Contingency / Expected / Residual Tables, Plot Generation, Result Saving, Summary JSON, và Output Registry** của riêng file này.